# 서울시 100m 격자·행정동 만족활동 기반 선호·잠재수요

이 노트북은 확정된 성별×연령별 **절대 선호확률**을 2024년 100m 격자별 문화누리 대상자 추정인구에 적용합니다. 모델을 다시 정의하지 않고 `src/preference_analysis`의 검증된 함수를 호출합니다. 접근성·가맹점·거리·이동시간은 사용하지 않습니다.

$$D_{g,c}=\sum_{s,a}N_{g,s,a}p_{s,a,c}$$

- $D_{g,c}$: 격자 $g$의 분야 $c$ 절대 잠재수요
- $N_{g,s,a}$: 격자의 성별 $s$·연령 $a$ 문화누리 대상자 추정인구
- $p_{s,a,c}$: 해당 집단의 분야 $c$ 절대 선호확률
- `preference_share_conditional_mnc`는 9개 정책 분야 내부의 상대구성 표시용이며 잠재수요 계산에는 사용하지 않습니다.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd
from IPython.display import FileLink, display

def find_project_root():
    configured = os.environ.get('ORACLE_PROJECT_ROOT')
    starts = ([Path(configured).expanduser()] if configured else []) + [Path.cwd()]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / '.git').exists() and (candidate / 'data').is_dir():
                return candidate.resolve()
    raise FileNotFoundError('Oracle-Project 저장소를 찾지 못했습니다.')

ROOT = find_project_root()
OUTPUT_DIR = ROOT / 'data/processed/preference_analysis/spatial'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('프로젝트:', ROOT)


프로젝트: .


## 1. 계산 실행

결과가 없을 때만 전체 공간 파이프라인을 실행합니다. 강제로 다시 계산하려면 `REBUILD=True`로 바꾸세요. HTML 지도는 계산된 CSV를 시각화한 정적 산출물이므로 지도를 열 때 모델이 다시 학습되지 않습니다.


In [2]:
REBUILD = False
required_output = OUTPUT_DIR / 'grid_middle_category_preference_demand_2024.csv'
if REBUILD or not required_output.exists():
    env = os.environ.copy()
    env['PYTHONPATH'] = str(ROOT / 'src')
    completed = subprocess.run(
        [sys.executable, '-m', 'preference_analysis.build_spatial_outputs', '--project-root', str(ROOT)],
        cwd=ROOT, env=env, capture_output=True, text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr[-5000:])
    print(completed.stdout)
else:
    print('✓ 기존 계산결과를 사용합니다. REBUILD=True이면 다시 계산합니다.')


✓ 기존 계산결과를 사용합니다. REBUILD=True이면 다시 계산합니다.


## 2. 핵심 결과와 정합성 검증

지역별 대상자 수는 분야마다 반복 저장되므로 분야를 가로질러 합산하면 안 됩니다. 절대 잠재수요는 분야별로 합산할 수 있으며, 정책 9개 분야 잠재수요와 기타 잠재수요 합은 전체 대상자 수와 일치해야 합니다.


In [3]:
grid = pd.read_csv(OUTPUT_DIR / 'grid_middle_category_preference_demand_2024.csv', encoding='utf-8-sig', low_memory=False)
dong = pd.read_csv(OUTPUT_DIR / 'dong_middle_category_preference_demand_2024.csv', encoding='utf-8-sig')
gu = pd.read_csv(OUTPUT_DIR / 'gu_middle_category_preference_demand_2024.csv', encoding='utf-8-sig')
validation = pd.read_csv(OUTPUT_DIR / 'spatial_validation_summary_2024.csv', encoding='utf-8-sig')

summary = pd.DataFrame({
    '항목': ['100m 격자', '행정동', '자치구', '15세 이상 추정 대상자'],
    '값': [
        f"{grid['GRID_CD'].nunique():,}개",
        f"{dong['행정동코드'].nunique():,}개",
        f"{gu['자치구코드'].nunique():,}개",
        f"{grid.drop_duplicates('GRID_CD')['target_population_est'].sum():,.0f}명",
    ],
})
display(summary)
display(validation)


,항목,값
0,100m 격자,"60,528개"
1,행정동,426개
2,자치구,25개
3,15세 이상 추정 대상자,"545,692명"


,check,status,value,detail
0,grid_policy_plus_other_equals_target,pass,-2.328306e-10,정책 9개 잠재수요와 기타 잠재수요 합의 대상자 총량 오차
1,grid_category_key_unique,pass,0.000000e+00,GRID_CD×분야 중복 행 수
2,zero_target_grids_are_no_data,pass,3.211900e+04,대상자 0명 격자는 확률 무자료·잠재수요 0으로 보존
3,grid_conditional_policy_share_sums_to_one,pass,3.330669e-16,대상자 양수 격자의 정책 9개 조건부 구성비 최대 절대오차
4,dong_target_conservation,pass,0.000000e+00,격자→dong 대상자 총량 오차
5,dong_potential_demand_conservation,pass,5.820766e-11,격자→dong 정책·기타 잠재수요 최대 총량 오차
6,gu_target_conservation,pass,0.000000e+00,격자→gu 대상자 총량 오차
7,gu_potential_demand_conservation,pass,0.000000e+00,격자→gu 정책·기타 잠재수요 최대 총량 오차


In [4]:
category_summary = (
    gu.groupby('middle_category', as_index=False)
      .agg(절대_잠재수요=('potential_demand_absolute', 'sum'))
      .sort_values('절대_잠재수요', ascending=False)
)
category_summary['절대_잠재수요'] = category_summary['절대_잠재수요'].round(1)
display(category_summary.rename(columns={'middle_category': '정책 분야'}))


,정책 분야,절대_잠재수요
1,관광지,102329.4
6,영상,95631.7
8,체육시설,76376.9
3,문화체험,12245.8
0,공연,9132.2
5,스포츠관람,8408.9
2,도서,7327.7
7,음악,6731.3
4,미술,5647.5


## 3. 외적 타당성

2024년 카드 이용건수·금액은 실제 이용이며, 만족활동 기반 예측선호와 동일한 값이 아닙니다. 차이는 모델 오차율이 아니라 접근성·공급·가격·정보·거래빈도 등이 반영된 **구성 차이**로 해석합니다. 이용건수 비교를 주 지표로, 이용금액 비교를 가격에 민감한 보조지표로 사용합니다. 카드 자치구 기준은 미확인이고, 문화예술조사 비교는 전국 표본의 성별×연령 방향성 점검입니다.


In [5]:
external_summary_path = OUTPUT_DIR / 'external_validation_2024_summary.csv'
if external_summary_path.exists():
    external_summary = pd.read_csv(external_summary_path, encoding='utf-8-sig')
    display(external_summary[['crosswalk_version', 'metric', 'value', 'interpretation']])
    print('주의: 카드 자치구는 이용자 거주지/가맹점 소재지 기준이 확인되지 않았습니다.')
    sensitivity = pd.read_csv(OUTPUT_DIR / 'external_validation_2024_crosswalk_sensitivity.csv', encoding='utf-8-sig')
    sensitivity = sensitivity[sensitivity['metric'].isin(['mapped9_transaction_coverage', 'seoul_transaction_spearman_rho_9categories', 'seoul_transaction_distribution_match_score'])]
    display(sensitivity[['crosswalk_version', 'metric', 'value']])
    arts_summary = pd.read_csv(OUTPUT_DIR / 'external_validation_arts_2024_summary.csv', encoding='utf-8-sig')
    display(arts_summary[['population_scope', 'middle_category', 'spearman_rho', 'interpretation']])
else:
    print('외적 타당성 결과가 없습니다. 전체 공간 파이프라인을 실행하세요.')


,crosswalk_version,metric,value,interpretation
0,primary_semantic_v1,mapped9_transaction_coverage,0.839702,전체 카드 이용건수 중 설정된 crosswalk로 매핑된 정책 9개 분야 비중
1,primary_semantic_v1,mapped9_amount_coverage,0.817334,전체 카드 이용금액 중 설정된 crosswalk로 매핑된 정책 9개 분야 비중
2,primary_semantic_v1,seoul_transaction_spearman_rho_9categories,0.483333,서울 전체 9개 분야 순위 일치 방향성
3,primary_semantic_v1,seoul_amount_spearman_rho_9categories,0.433333,서울 전체 9개 분야 금액 순위 일치 방향성
4,primary_semantic_v1,seoul_transaction_distribution_match_score,43.915743,예측 조건부 구성과 카드 거래 구성의 기술적 분포 일치도; 모델 성능점수가 아님
5,primary_semantic_v1,seoul_amount_distribution_match_score,40.027010,예측 조건부 구성과 카드 금액 구성의 기술적 분포 일치도; 모델 성능점수가 아님


주의: 카드 자치구는 이용자 거주지/가맹점 소재지 기준이 확인되지 않았습니다.


,crosswalk_version,metric,value
0,primary_semantic_v1,mapped9_transaction_coverage,0.839702
2,primary_semantic_v1,seoul_transaction_spearman_rho_9categories,0.483333
4,primary_semantic_v1,seoul_transaction_distribution_match_score,43.915743
6,legacy_eda_craft_as_art_v1,mapped9_transaction_coverage,0.839702
8,legacy_eda_craft_as_art_v1,seoul_transaction_spearman_rho_9categories,0.166667
10,legacy_eda_craft_as_art_v1,seoul_transaction_distribution_match_score,45.072206
12,conservative_exclude_craft_general_v1,mapped9_transaction_coverage,0.736573
14,conservative_exclude_craft_general_v1,seoul_transaction_spearman_rho_9categories,0.533333
16,conservative_exclude_craft_general_v1,seoul_transaction_distribution_match_score,43.437778


,population_scope,middle_category,spearman_rho,interpretation
0,전국 조사표본,공연,0.854945,공연 직접관람 여부와 만족활동 선호의 성별×연령 방향성 비교
1,전국 조사표본,미술,0.947253,미술전시 직접관람 여부와 만족활동 선호의 성별×연령 방향성 비교
2,전국 조사표본,영상,0.112088,영화관람만 비교하므로 TV·OTT를 포함한 영상 분야와 부분 비교


## 4. 인터랙티브 지도

각 지도에서 9개 정책 분야와 `절대 선호확률/절대 잠재수요`를 선택할 수 있습니다. 격자 또는 행정동을 클릭하면 대상자 수·절대확률·잠재수요·조건부 구성비·기타확률이 표시됩니다. 대상자 0명 격자는 0%가 아닌 무자료로 표시됩니다. 색 구간은 대상자 양수 지역의 분야별 분위이므로 분야 간 색 농도를 직접 비교하지 않습니다.


In [6]:
grid_map = OUTPUT_DIR / 'maps/grid_preference_demand_2024.html'
dong_map = OUTPUT_DIR / 'maps/dong_preference_demand_2024.html'
print('100m 격자 지도')
display(FileLink(grid_map))
print('행정동 지도')
display(FileLink(dong_map))


100m 격자 지도


data/processed/preference_analysis/spatial/maps/grid_preference_demand_2024.html

행정동 지도


data/processed/preference_analysis/spatial/maps/dong_preference_demand_2024.html

## 5. 해석상 한계

- 결과는 국민여가활동조사의 만족활동을 기반으로 한 경험선호 추정치입니다. 미래 희망이나 실제 이용자 수가 아닙니다.
- 100m 값은 성별×연령별 대상자 추정인구에 기반하므로 작은 격자의 불확실성이 큽니다. 정책 설명에는 행정동 결과를 함께 사용합니다.
- 2024 추정치를 2025년 단순화 행정동 경계에 코드로 결합해 표시하지만, 격자 소속 행정동은 기존 2024 분석 연결표를 유지합니다.
- 가구소득은 격자별 성별×연령×소득 결합인구가 없어 공간 적용 주모형에서 제외했습니다.
- 카드 자치구가 이용자 거주지 기준인지 이용 가맹점 소재지 기준인지 확인되지 않아 지역 외부검증은 방향성 점검입니다.
- 국민문화예술활동조사 비교는 2024년 전국 조사표본이며 서울 표본 검증이 아닙니다.
- 접근성과 가맹점 공급은 이후 별도 파트에서 결합합니다.
